# 04. Multi-Query Retrieval - AI 트렌드 보고서 기반
## 개념

Multi-Query Retrieval은 하나의 질문을 **여러 관점의 검색 쿼리**로 확장한 뒤 각각 검색하고, 검색 결과를 합쳐 더 풍부한 context를 구성하는 방법입니다.
AI 트렌드 보고서는 `AI 산업`, `AI 기술`, `AI 정책`, `2026년 전망`, `토픽모델링`, `추론형 데이터`처럼 섹션과 용어가 명확합니다. 하지만 사용자의 질문은 보통 짧고 구어체입니다.
따라서 하나의 질문을 여러 검색 쿼리로 확장하면, 단일 쿼리로 놓칠 수 있는 관련 문서를 더 넓게 찾을 수 있습니다.
검색 누락이 자주 발생하는 RAG 시스템에서 Recall을 높이기 위해 선택적으로 쓰는 기법입니다.

핵심 판단

```text
원본 질문: "AI가 회사 일에 어떻게 들어와요?"

생성된 쿼리 예시:
  1. 2026년 AI 산업 전망에서 기업 운영 인프라와 AI 에이전트 도입은 어떻게 설명되는가
  2. AI 에이전트가 문서 처리, 고객 지원, 운영 자동화에 미치는 영향은 무엇인가
  3. AI 산업 토픽에서 인프라·에이전트 기반 운영 모델 변화는 어떤 의미인가

각 쿼리로 검색 → 결과 합산 → 중복 제거 → 더 풍부한 context 구성
```

## 실습 목표

1. LangChain 내장 `MultiQueryRetriever`를 사용해 다중 쿼리 검색을 수행한다.
2. 직접 다중 쿼리를 생성하고 검색 결과를 중복 제거한다.
3. RAG Fusion 방식으로 여러 검색 결과의 순위를 통합한다.
4. Multi-Query 기반 RAG 체인을 완성한다.


## Markdown 기반 VectorDB 불러오기
이번 실습에서는 02번에서 생성한 Markdown 기반 ChromaDB를 사용한다.  
따라서 이 노트북을 실행하기 전에 `02_data_pdf_to_md.ipynb`를 먼저 실행하여 `chroma_db/ai_trend_report_md`가 생성되어 있어야 한다.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 02_data_pdf_to_md.ipynb에서 생성한 Markdown 기반 ChromaDB를 불러온다.
# AI@Data Report: 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망

DB_PATH = Path("chroma_db/ai_trend_report_md")
COLLECTION_NAME = "ai_trend_report_2026_md"
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print(f"불러온 청크 수: {vector_store._collection.count()}")
print("Vector Store 로딩 완료")


불러온 청크 수: 27
Vector Store 로딩 완료


## 검색 결과 확인 함수
- 검색된 문서 목록을 간단히 확인하기 위한 출력 함수 정의

In [2]:
def preview_docs(docs, max_chars: int = 250):
    """검색된 문서 목록을 간단히 확인하기 위한 출력 함수"""
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        doc_type = doc.metadata.get("doc_type", "?")
        source_name = doc.metadata.get("source_name", "?")

        print(f"\n[{i}] source={source}")
        print(f"chunk_id={chunk_id} | doc_type={doc_type} | source_name={source_name}")
        print("-" * 80)
        print(doc.page_content[:max_chars])
        print("-" * 80)

## MultiQueryRetriever: LangChain 내장 기능

먼저 LangChain에서 제공하는 `MultiQueryRetriever`를 사용해 다중 쿼리 검색을 실행한다.  
이 방식은 LLM이 원본 질문을 여러 검색 질문으로 자동 변환하고, 각 질문으로 검색한 결과를 합쳐 반환한다.

이 단계에서는 직접 구현보다 “Multi-Query Retrieval이 어떤 방식으로 동작하는지”를 확인하는 데 초점을 둔다.


```text
사용자 : 질문 AI가 회사 일에 어떻게 들어와요?

[LLM이 생성한 질문 예시]
검색 질문 1:
AI 에이전트가 기업 업무 자동화에 미치는 영향은 무엇인가?

검색 질문 2:
생성형 AI가 문서 처리, 고객 지원, 운영 프로세스에 어떻게 활용되는가?

검색 질문 3:
2026년 AI 산업 전망에서 기업 운영 방식 변화는 어떻게 설명되는가?
```

In [12]:
from langchain_classic.retrievers import MultiQueryRetriever

# 멀티 쿼리 생성값을 확인
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s",
    force=True
)

logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_retriever = MultiQueryRetriever.from_llm(
    # base_retriever의 k=3은 각 검색 쿼리마다 최대 3개의 문서를 가져온다는 의미이다.
    # 최종 문서 수는 생성된 쿼리 수와 중복 제거 결과에 따라 달라진다.
    retriever=base_retriever,   
    llm=llm
)

question = "AI가 회사 일에 어떻게 들어와요?"
results = multi_retriever.invoke(question)

print(f"원본 질문: {question}")
print(f"검색된 문서 수(중복 제거 후): {len(results)}")
preview_docs(results, max_chars=220)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['AI가 기업 운영에 어떻게 적용될 수 있나요?  ', 'AI는 비즈니스 프로세스에 어떤 방식으로 통합되나요?  ', 'AI 기술이 회사의 업무에 미치는 영향은 무엇인가요?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


원본 질문: AI가 회사 일에 어떻게 들어와요?
검색된 문서 수(중복 제거 후): 3

[1] source=data\AI@Data_Report_CLEANED.md
chunk_id=21 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
--------------------------------------------------------------------------------
기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며

– –
사람 에이전트 시스템이 혼합된 업무 구조 가 일부 영역에서 확산될 가능성이 있음

-
기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할

것으로 예상되지만, 그 영향은 아직 초기 단계에 머물며 중장기적 구조 변화의 단초를 형성하는

수준으로 평가됨

--------------------------------------------------------------------------------

[2] source=data\AI@Data_Report_CLEANED.md
chunk_id=20 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
--------------------------------------------------------------------------------
구조적 흐름을 도출하였음

- 분석 결과를 통해 세 분야는 서로 다른 영역에서 출발하더라도 활용 확대 → 기술적 고도화

요구 증가 → 제도적 대응 심화 로 이어지는 연속적 흐름을 형성하고 있음을 확인하였음

2026년에는 이 세 방향성이 더욱 맞물려 산업 구조의 재배치, 기술 체계의 정교화,

규제·책임 체계의 본격적 적용으로 이어지는 개략적 전망을 제시함

2026년 AI 시장은 선도 기
-------------------------------------

In [ ]:
# 원본 질문 1개 + 생성 쿼리 3개 = 총 4개 질문
# 총 4개 질문 × 각 질문당 Top-3 검색 = 최대 12개 chunk 검색

## 단일 쿼리 검색 vs 다중 쿼리 비교
검색 결과 수가 많다는 것은 더 많은 후보 문서를 확보했다는 의미이다.  
하지만 문서 수가 많다고 항상 답변 품질이 좋아지는 것은 아니다.  
이 단계에서는 단일 쿼리와 다중 쿼리가 가져오는 문서의 범위와 다양성을 비교한다.

In [4]:
# 단일 쿼리 검색과 다중 쿼리 검색 결과 수 비교
single_results = base_retriever.invoke(question)
multi_results = multi_retriever.invoke(question)

print(f"단일 쿼리 검색 결과 수: {len(single_results)}개")
print(f"다중 쿼리 검색 결과 수: {len(multi_results)}개")

print("\n[단일 쿼리 검색 결과]")
preview_docs(single_results, max_chars=180)

print("\n[다중 쿼리 검색 결과]")
preview_docs(multi_results, max_chars=180)


단일 쿼리 검색 결과 수: 3개
다중 쿼리 검색 결과 수: 3개

[단일 쿼리 검색 결과]

[1] source=data\AI@Data_Report_CLEANED.md
chunk_id=27 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
--------------------------------------------------------------------------------
전문가·산업계 의견을 주기적으로 수렴하고, 이를 관계부처 정책 검토로 연계할 필요가 있음

글로벌 AI 규제 환경과 국내 법·제도 간의 정합성을 지속적으로 점검하여, AI 혁신을 저해하지

않으면서도 저작권 보호와 규제 안정성을 함께 확보할 수 있는 정책 논의 필요

---

#### **토픽 분석을 통한** **AI 주
--------------------------------------------------------------------------------

[2] source=data\AI@Data_Report_CLEANED.md
chunk_id=7 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
--------------------------------------------------------------------------------
2025년 AI 생태계에서 부상하는 핵심 이슈와 구조적 변화 를 입체적으로 파악하고자 함

- 생성형 AI 고도화, AI 인프라 경쟁, 안전·책임성 강화, 국제 기준 정비 등 주요 변화 요인을

식별하여 산업·기술·정책 분야의 변화 방향성을 명확히 도출함

더 나아가 분석 결과를 기반으로 국가 정책, 기업 전략, 기술 개
--------------------------------------------------------------------------------

[3] source=data\AI@Data_Report_CL

## [참고] 직접 구현 - 쿼리 생성 + 검색 + 중복 제거

이번에는 `MultiQueryRetriever`를 사용하지 않고 직접 구현합니다.

흐름은 다음과 같습니다.

```text
원본 질문
  ↓
LLM으로 여러 검색 쿼리 생성
  ↓
원본 질문 + 생성된 쿼리들로 각각 검색
  ↓
각 검색 쿼리마다 최대 k개의 문서 검색
  ↓    
검색 결과를 하나로 합침
  ↓
중복 문서 제거
```


예를 들어 생성 쿼리가 3개이고 `k=3`이면, 생성 쿼리 기준으로 최대 9개의 문서가 검색될 수 있다.  
원본 질문까지 함께 검색하면 최대 12개의 문서가 수집될 수 있다.  
다만 같은 문서가 여러 쿼리에서 반복 검색될 수 있으므로, 중복 제거 후 최종 문서 수는 더 적어질 수 있다.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# AI 트렌드 보고서 특화 다중 쿼리 생성 프롬프트
multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 트렌드 보고서 기반 RAG 검색 전문가입니다.
주어진 질문을 아래 3가지 관점에서 검색 쿼리로 변환하세요.

- 산업 관점: AI 도입, 시장 규모, 기업 운영, AI 에이전트, 인프라
- 기술 관점: 멀티모달, 추론형 AI, 온디바이스 AI, 모델 고도화, 합성데이터
- 정책 관점: AI 규제, 안전성, 책임성, 거버넌스, AI 기본법, EU AI Act

각 쿼리를 줄바꿈으로 구분하고, 번호나 설명 없이 쿼리만 작성하세요."""),
    ("human", "{question}")
])

query_gen_chain = multi_query_prompt | llm | StrOutputParser()

question = "AI가 회사 일에 어떻게 들어와요?"
generated = query_gen_chain.invoke({"question": question})

queries = [q.strip() for q in generated.strip().split("\n") if q.strip()]

print(f"원본 질문: {question}")
print("\n생성된 쿼리:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")


원본 질문: AI가 회사 일에 어떻게 들어와요?

생성된 쿼리:
  1. AI 도입 기업 운영 사례
  2. AI 시장 규모 및 성장 전망
  3. AI 에이전트 활용 사례 및 효과
  4. AI 인프라 구축 및 운영 전략
  5. AI 도입에 따른 기업의 변화
  6. 멀티모달 AI 기술 적용 사례
  7. 추론형 AI의 비즈니스 활용
  8. 온디바이스 AI의 장점과 사례
  9. 모델 고도화 기술 동향
  10. 합성데이터 생성 및 활용 방법
  11. AI 규제 현황 및 기업 대응 방안
  12. AI 안전성 확보를 위한 정책
  13. AI 책임성 및 윤리적 고려사항
  14. AI 거버넌스 체계 구축 방안
  15. AI 기본법 및 EU AI Act의 영향


In [13]:
from langchain_core.documents import Document

def multi_query_retrieve(question: str, retriever) -> list[Document]:
    """다중 쿼리를 생성한 뒤 검색하고, 중복 문서를 제거하여 반환한다."""
    generated = query_gen_chain.invoke({"question": question})
    generated_queries = [q.strip() for q in generated.strip().split("\n") if q.strip()]

    # 원본 질문도 검색에 포함한다.
    queries = [question] + generated_queries

    seen = set()
    all_docs = []

    for q in queries:
        docs = retriever.invoke(q)
        for doc in docs:
            # 같은 chunk가 여러 쿼리에서 반복 검색될 수 있으므로 page_content 기준으로 중복 제거한다.
            key = doc.page_content
            if key not in seen:
                seen.add(key)
                all_docs.append(doc)

    return all_docs


base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
results = multi_query_retrieve(question, base_retriever)

print(f"수집된 문서 수(중복 제거 후): {len(results)}")
preview_docs(results, max_chars=220)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/emb

수집된 문서 수(중복 제거 후): 18

[1] source=data\AI@Data_Report_CLEANED.md
chunk_id=27 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
--------------------------------------------------------------------------------
전문가·산업계 의견을 주기적으로 수렴하고, 이를 관계부처 정책 검토로 연계할 필요가 있음

글로벌 AI 규제 환경과 국내 법·제도 간의 정합성을 지속적으로 점검하여, AI 혁신을 저해하지

않으면서도 저작권 보호와 규제 안정성을 함께 확보할 수 있는 정책 논의 필요

---

#### **토픽 분석을 통한** **AI 주요 트렌드 및 2026 전망**

## AI@Data Report

발 
--------------------------------------------------------------------------------

[2] source=data\AI@Data_Report_CLEANED.md
chunk_id=7 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
--------------------------------------------------------------------------------
2025년 AI 생태계에서 부상하는 핵심 이슈와 구조적 변화 를 입체적으로 파악하고자 함

- 생성형 AI 고도화, AI 인프라 경쟁, 안전·책임성 강화, 국제 기준 정비 등 주요 변화 요인을

식별하여 산업·기술·정책 분야의 변화 방향성을 명확히 도출함

더 나아가 분석 결과를 기반으로 국가 정책, 기업 전략, 기술 개발 방향 설정에 활용

가능한 실질적 인사이트를 제공하고, 중장기 전략 
----------------------------------------------------------------

In [14]:
# 원본 질문 1개 + 생성 쿼리 15개 = 총 16개 검색 쿼리
# k = 3이면,
# 16개 검색 쿼리 × 각 쿼리당 3개 chunk = 최대 48개 chunk 후보 
# 중복 제거후 : 18개

## 확장 실습: RAG Fusion-RRF 기반 문서 통합

단순히 검색 결과를 합치면 순위 정보가 사라질 수 있다.

RAG Fusion은 여러 쿼리의 검색 결과를 통합할 때 **각 검색 결과에서 높은 순위로 나온 문서에 더 높은 점수**를 부여한다.

여기서는 대표적인 방식인 **RRF(Reciprocal Rank Fusion)** 를 사용한다.

```text
→ LLM으로 검색 질문 3개 생성
→ 각 질문별로 검색
→ 검색 결과 리스트 여러 개 생성
→ RRF 점수로 문서 순위 재계산
→ 최종 문서 선택
```
`MultiQueryRetriever`는 여러 쿼리의 검색 결과를 자동으로 합쳐 반환한다.  
다만 검색 결과의 순위를 더 정교하게 조정하고 싶다면, 직접 Multi-Query를 구현한 뒤 RRF 방식으로 문서 순위를 다시 계산할 수 있다.


### RAG Fusion 실행: RRF로 검색 결과 통합

In [ ]:
def reciprocal_rank_fusion(results_list: list[list[Document]], rrf_k: int = 60) -> list[Document]:
    """여러 검색 결과를 RRF 알고리즘으로 통합한다."""
    scores = {}
    doc_map = {}
  
    for results in results_list:
        for rank, doc in enumerate(results, start=1):
            key = doc.page_content
            scores[key] = scores.get(key, 0) + 1 / (rrf_k + rank)  # RRF 수식 1 / (rrf_k + rank + 1)
            doc_map[key] = doc

    sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)
    return [doc_map[key] for key in sorted_keys]


# 여러 쿼리 생성
generated = query_gen_chain.invoke({"question": question})
generated_queries = [q.strip() for q in generated.strip().split("\n") if q.strip()]
queries = [question] + generated_queries

print("[검색에 사용할 쿼리]")
for i, q in enumerate(queries, start=1):
    print(f"{i}. {q}")

# 각 쿼리로 검색한 뒤 RRF로 통합
all_results = [base_retriever.invoke(q) for q in queries]
fused = reciprocal_rank_fusion(all_results)

print(f"\nRAG Fusion 결과: {len(fused)}개")
preview_docs(fused[:5], max_chars=220)


[검색에 사용할 쿼리]
1. AI가 회사 일에 어떻게 들어와요?
2. AI 도입 기업 운영 사례
3. AI 시장 규모 및 성장 전망
4. AI 에이전트 활용 사례 및 효과
5. AI 인프라 구축 및 운영 전략
6. 멀티모달 AI 기술 적용 사례
7. 추론형 AI의 비즈니스 활용
8. 온디바이스 AI의 장점과 사례
9. 모델 고도화 기술 동향
10. 합성데이터 생성 및 활용 방법
11. AI 규제 현황 및 기업 대응 방안
12. AI 안전성 확보를 위한 정책
13. AI 책임성 및 윤리적 고려사항
14. AI 거버넌스 체계 구축 방안
15. AI 기본법 및 EU AI Act의 영향

RAG Fusion 결과: 21개

[1] source=data\AI@Data_Report_CLEANED.md
chunk_id=22 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
--------------------------------------------------------------------------------
- - 이와 함께 기존 산업의 가치 사슬도 전면 재편이라기보다는 핵심 공정·고부가 서비스 단에서 데이터·AI 중심 구조로 이동하는 압력이 강화되는 양상을 보일 것으로 전망됨 

- 기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며 – – 

- 사람 에이전트 시스템이 혼합된 업무 구조 가 일부 영역에서 확산될 가능성이 있음 

- - 기업 간(B2B) 
--------------------------------------------------------------------------------

[2] source=data\AI@Data_Report_CLEANED.md
chunk_id=6 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
----------------------------------------

## Multi-Query RAG 체인 완성: 중복 제거 기반 방식

주의: 아래 최종 RAG 체인은 앞에서 실습한 RRF 기반 RAG Fusion을 적용하지 않는다.  
직접 구현한 `multi_query_retrieve()` 함수를 사용하여 여러 쿼리의 검색 결과를 모은 뒤, 중복 문서를 제거하여 context를 구성한다.

실행 흐름은 다음과 같습니다.

```text
입력 질문
  ↓
여러 검색 쿼리 생성
  ↓
각 쿼리로 Vector Store 검색
  ↓
검색 결과 중복 제거
  ↓
context 생성
  ↓
원본 질문 + context를 프롬프트에 입력
  ↓
LLM 답변 생성
```


In [15]:
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    """검색된 문서들을 하나의 context 문자열로 합친다."""
    return "\n\n".join(doc.page_content for doc in docs)


answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서에 근거해서만 질문에 답하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.
한 줄에 한 문장씩 깔끔하게 정리하세요.

[문서]
{context}"""),
    ("human", "{question}")
])

multi_rag_chain = (
    # 1. 원본 질문을 유지하면서 Multi-Query 검색 결과를 context로 추가한다.
    RunnablePassthrough.assign(
        context=lambda x: format_docs(
            multi_query_retrieve(x["question"], base_retriever)
        )
    )

    # 2. 원본 질문과 검색된 context를 프롬프트에 넣는다.
    | answer_prompt

    # 3. LLM이 문서 기반 답변을 생성한다.
    | llm

    # 4. 최종 답변을 문자열로 변환한다.
    | StrOutputParser()
)

answer = multi_rag_chain.invoke({
    "question": "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"
})

print(answer)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026년 AI 기술 전망에서 핵심 변화는 지능 구조 고도화와 데이터 한계 보완입니다. 

합성데이터, 추론형 AI, 멀티모달 기술이 주요 경쟁 축으로 자리 잡을 것입니다. 

이러한 기술 고도화는 학습 효율 향상, 복합 정보 처리, 설명 가능성 강화를 가져올 것으로 예상됩니다. 

AI의 활용 가능성이 넓어지고, 복잡한 상황에서의 판단 능력이 강화될 것입니다. 

안전성 및 정합성 검증의 중요성이 커지면서 신뢰성 평가와 표준화가 필요해질 것입니다.


## 추가 질문으로 Multi-Query 검색 확인

이번에는 표 기반 질문, 정책 질문, 산업 질문을 사용해 Multi-Query 검색 결과를 추가로 확인한다.  
질문 유형에 따라 생성되는 검색 쿼리와 검색 결과가 어떻게 달라지는지 비교한다.

확인할 질문은 다음과 같다.

1. AI 산업, AI 기술, AI 정책의 핵심 이슈 비교
2. AI 규제와 안전성의 중요성
3. 추론형 데이터의 유형 구분

In [14]:
# 추가 테스트 질문
# 표 기반 질문, 정책 질문, 산업 질문에서 Multi-Query 검색이 어떤 차이를 보이는지 확인한다.

test_questions = [
    "AI 산업, AI 기술, AI 정책의 핵심 이슈를 비교해줘.",
    "AI 규제와 안전성은 왜 중요해졌나요?",
    "추론형 데이터는 어떤 유형으로 나뉘나요?",
]

for q in test_questions:
    print("=" * 100)
    print(f"[질문] {q}")
    print(multi_rag_chain.invoke({"question": q}))
    print()


[질문] AI 산업, AI 기술, AI 정책의 핵심 이슈를 비교해줘.
AI 산업의 핵심 이슈는 AI 도입·확산 가속에 따른 산업 구조 변화와 글로벌 경쟁 심화입니다.  
AI 기술의 핵심 이슈는 지능 구조 고도화 중심의 기술 발전 흐름과 성능 고도화입니다.  
AI 정책의 핵심 이슈는 안전성 확보와 위험 관리 중심 정책, 그리고 책임·의무 강화와 신뢰 기반 거버넌스 확립입니다.

[질문] AI 규제와 안전성은 왜 중요해졌나요?
AI 규제와 안전성이 중요해진 이유는 다음과 같습니다. 

AI의 확산 속도 대비 위험 관리와 안전 확보 체계를 시급히 강화해야 한다는 정책적 요구가 높아지고 있습니다. 

AI 관련 사고와 위험 보고 건수가 지속적으로 증가하고 있으며, 특히 2023~2024년 이후 가파른 상승 추세를 보이고 있습니다. 

산업·정책 영역에서는 제도화 지연에 따른 불확실성과 대응 부담이 증가하고 있습니다. 

AI 활용과 규제 준수 사이에서 복합적 리스크에 직면하고 있는 국가와 기업이 많아지고 있습니다. 

안전성 확보와 위험 관리 중심의 정책이 필요하다는 인식이 확산되고 있습니다. 

AI 기술의 발전과 시장 확산이 제도·규범 정비보다 앞서 나가면서 규제 공백이 커지는 경향이 나타나고 있습니다. 

따라서 AI 규제와 안전성은 AI의 지속 가능한 발전과 사회적 신뢰를 확보하기 위해 필수적입니다.

[질문] 추론형 데이터는 어떤 유형으로 나뉘나요?
추론형 데이터는 다음과 같은 유형으로 나뉩니다.

1. 단계적 '과정' 추론
2. '근거' 기반 설명
3. '맥락·관계' 이해
4. 논리·취약점 검증



# 정리

| 방법 | 특징 | 수업에서 확인할 점 |
|---|---|---|
| 단일 쿼리 검색 | 사용자의 질문 하나로만 검색 | 질문 표현에 따라 검색 결과가 달라질 수 있음 |
| MultiQueryRetriever | LangChain 내장 방식으로 여러 쿼리 자동 생성 | 간편하지만 생성 쿼리를 세밀하게 제어하기는 어려움 |
| 직접 구현 | 프롬프트로 쿼리 생성 관점을 직접 설계 | 도메인에 맞는 검색 전략을 만들 수 있음 |
| RAG Fusion (RRF) | 여러 검색 결과의 순위를 통합 | 여러 쿼리에서 반복적으로 높은 순위에 나온 문서를 우선 활용 |

# 실습 확인 포인트

- 단일 쿼리 검색과 다중 쿼리 검색의 결과가 어떻게 다른지 확인한다.
- 생성된 쿼리가 `AI 산업`, `AI 기술`, `AI 정책`, `2026년 전망` 등 보고서 용어를 잘 반영하는지 확인한다.
- 다중 쿼리 검색이 관련 문서를 더 다양하게 가져오는지 확인한다.
- 중복 제거 후에도 답변에 필요한 근거가 충분히 남는지 확인한다.
- ChromaDB의 score는 거리값이므로 낮을수록 더 유사하다고 해석한다.
- PDF 직접 로딩 기반 DB와 Markdown 기반 DB의 score 절대값은 직접 비교하지 않는다.

